# SuAVE Image to Label SVM Prediction Model

In [ ]:
# -- Colab bootstrap (no-op on Binder / JupyterHub) ------------------------
import os, sys
if "google.colab" in sys.modules:
    import subprocess
    if os.path.isdir("/tmp/suave-nb/.git"):
        subprocess.run(["git","-C","/tmp/suave-nb","fetch","--depth=1","origin","main"],
                       capture_output=True)
        subprocess.run(["git","-C","/tmp/suave-nb","reset","--hard","origin/main"],
                       capture_output=True)
    else:
        _r = subprocess.run(
            ["git","clone","--depth=1",
             "https://github.com/izaslavsky/suave-notebooks.git","/tmp/suave-nb"],
            capture_output=True, text=True)
        if _r.returncode != 0:
            raise RuntimeError(f"Could not clone suave-notebooks:\n{_r.stderr}")
    sys.path.insert(0, "/tmp/suave-nb/helpers")


In [ ]:
# ── Colab only: mount Google Drive to load session credentials ─────────────
import sys
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')


In [ ]:
# ── Load SuAVE parameters ──────────────────────────────────────────────────
import sys, os, requests as _req, urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
from urllib.parse import urlparse as _urlparse
sys.path.insert(0, '../../helpers')
import suave_utils as su
from IPython.display import display, Markdown
import pandas as pd

def printmd(string):
    display(Markdown(string))

if not su.ENV.colab:
    # Binder / JupyterHub: parameters load automatically
    _p = su.load_params()
    if _p is None:
        from IPython.display import HTML as _HTML
        display(_HTML(
            '<div style="background:#dc3545;color:white;padding:14px 16px;'
            'border-radius:6px;font-size:15px;font-weight:bold;line-height:1.6;margin:4px 0">'
            '&#9888; No SuAVE parameters found.'
            '<br><span style="font-size:12px;font-weight:normal">'
            'Open this notebook via SuAVEDispatch from your survey in SuAVE.'
            '</span></div>'))
        class _Stop(Exception):
            def _render_traceback_(self): return []
        raise _Stop()
else:
    # Colab: try Drive/cache silently; credentials cell below handles fallback
    _p = su.load_params(_silent=True)


In [ ]:
# -- Colab: verify session loaded from Drive/cache --------------------------
from IPython.display import HTML
if su.ENV.colab:
    if _p is None:
        display(HTML(
            '<div style="background:#dc3545;color:white;padding:14px 16px;'
            'border-radius:6px;font-size:15px;font-weight:bold;line-height:1.6;margin:4px 0">'
            '&#9888; No SuAVE session found.'
            '<br><span style="font-size:12px;font-weight:normal">'
            'Open SuAVEDispatch from the correct survey in SuAVE, then re-open this notebook.'
            '</span></div>'))
        class _Stop(Exception):
            def _render_traceback_(self): return []
        raise _Stop()
    display(HTML(
        '<div style="font-size:12px;padding:8px 12px;border-radius:4px;margin:3px 0;'
        'background:#e8f5e9;border:1px solid #81c784">'
        'Session loaded &mdash; survey <b>' + _p.get('survey', '?') + '</b>'
        ', user <b>' + _p.get('user', '?') + '</b>.'
        '<br><span style="color:#666;font-size:11px">'
        'Wrong survey? Go back to SuAVE, open the correct survey, and click Jupyter again.'
        '</span></div>'))
else:
    su._skipped('Colab only')


In [ ]:
# ── Variables used throughout this notebook ────────────────────────────────
if _p is None:
    from IPython.display import HTML as _HTML
    display(_HTML(
        '<div style="background:#dc3545;color:white;padding:14px 16px;'
        'border-radius:6px;font-size:15px;font-weight:bold;line-height:1.6;margin:4px 0">'
        '&#9888; Parameters not loaded.'
        '<br><span style="font-size:12px;font-weight:normal">'
        'Navigate to your survey in SuAVE and relaunch this notebook to load parameters.'
        '</span></div>'))
    class _Stop(Exception):
        def _render_traceback_(self): return []
    raise _Stop()
# Variables used throughout this notebook
user          = _p.get('user', '')
survey_url    = _p.get('surveyurl', '')
csv_file      = _p.get('csv', '')
dzc_file      = _p.get('dzc', '')
active_object = _p.get('activeobject', 'none')
_caps         = su.detect_capabilities(_p)
views         = ','.join(_caps.get('views', []))
view          = 'grid'

# NFS image paths (non-empty only when running on a hub with NFS storage)
localdzc             = _caps.get('localdzc', '')
full_images          = _caps.get('full_images', '')
full_images_location = full_images   # alias used by some notebooks

# Fetch and cache survey CSV for panellibs.extract_data()
absolutePath = os.path.expanduser('~/temp_csvs/')
os.makedirs(absolutePath, exist_ok=True)
_origin = _urlparse(survey_url)
_csv_url = f"{_origin.scheme}://{_origin.netloc}/surveys/{csv_file}"
_r = _req.get(_csv_url, timeout=30, verify=False)
_r.raise_for_status()
with open(absolutePath + csv_file, 'wb') as _f:
    _f.write(_r.content)

su.show_params(_p)


### Import all packages

In [28]:
from __future__ import print_function
import warnings
warnings.filterwarnings('ignore')  # "error", "ignore", "always", "default", "module" or "once"

# More imports
import matplotlib.pyplot as plt
import numpy as np
import argparse
import random
import csv
import pandas as pd
import re
import cv2
import os

# import the necessary packages for SVM predictor
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
import imutils
import joblib

# import histogram func
from histogram import Histograms

# Import widget functionality
from ipywidgets import interact, interactive, fixed, interact_manual
import ipywidgets as widgets


import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

## Run and choose the column name that coresponds with the column label to be predicted

In [29]:
# use csv file to grab images/labels
csv_path = absolutePath + csv_file
df = pd.read_csv(csv_path)

localdzc = dzc_file.replace("https://maxim.ucsd.edu/dzgen/lib-staging-uploads","/lib-nfs/dzgen")
full_images_location = localdzc.replace("/content.dzc","/full_images/")

from bs4 import BeautifulSoup
from urllib.parse import urljoin

response = requests.get(full_images_location)
soup = BeautifulSoup(response.text, 'html.parser')

image_links = soup.find_all('a', href=True)

image_urls = []

def get_string_before_last_period(input_string):
    parts = input_string.rsplit('.', 1)
    if len(parts) > 1:
        return parts[0]
    else:
        return input_string

images_in_df = list(df["#img"].unique())

for link in image_links:
    image_url = link['href']
    if get_string_before_last_period(image_url) in images_in_df:
    
        if not image_url.startswith(('http:', 'https:')):
            full_image_url = urljoin(full_images_location, image_url)
        else:
            full_image_url = image_url

        image_urls.append(full_image_url)

# Choose column of label for prediction
toPredict = list(df.columns.values)

pred_menu = {}
for i in range(0, len(toPredict)):
    pred_menu[toPredict[i]] = toPredict[i]

def f(predictions_menu):
    return predictions_menu

out2 = interact(f, predictions_menu=pred_menu)

## Choose type of histogram to extract from images

In [30]:
hist_type = { 
    'HSV Color Histogram': 0,
    'Blue Histogram': 1,
    'Red Histogram': 2,
    'Green Histogram': 3
    }

def f(histogram_type):
    return histogram_type
typeH= interact(f, histogram_type=hist_type);

## Get histogram from each image in the dataset

In [31]:
# init the image suffix
suffix = '.jpg'
labels = []

labelHeader = out2.widget.result

# grab chosen column names
nameCol = df['#img']
predCol = df[labelHeader]

# add all fabric columns to the y set
for i in range (0,len(predCol)):
    labels.append(predCol[i])
# create new hist_list
hist_list = []
# gather images from path created from file names in csv file
for i in range(len(nameCol)):
    image_name = nameCol[i]
    url = full_images_location + image_name + ".png"
    response = requests.get(url)

    if response.status_code == 200:
        # Read the image from the response content
        image_bytes = np.frombuffer(response.content, np.uint8)
        im = cv2.imdecode(image_bytes, cv2.IMREAD_COLOR)
    else:
        print(f"Failed to fetch image from URL: {url}")
    if typeH == 0:
        hist = Histograms.extract_color_histogram(im)
    elif typeH == 1: 
        hist = Histograms.extract_blue_histogram(im)
    elif typeH == 2: 
        hist = Histograms.extract_red_histogram(im)
    elif typeH == 3: 
        hist = Histograms.extract_green_histogram(im)
    hist_list.append(hist)
    
# transform labels into numerical system
le = LabelEncoder()
labels = le.fit_transform(predCol)



# separate data into test/train sets for features/labels
(xtrain, xtest, ytrain, ytest) = train_test_split(np.array(hist_list),labels, test_size = 0.5)

# Train the linear regression classifier
model2 = LinearSVC()
model2.fit(xtrain, ytrain)

# Calculate predictions on the data set
predictions = model2.predict(np.array(hist_list))
target_names = [str(class_name) for class_name in le.classes_]

print(classification_report(labels, predictions, target_names=target_names))

In [36]:
# save the label
filePath = "labels/"
fileName = labelHeader + '_' + csv_file[:-4] + '_svmModel' + '.txt'
file = open(os.path.join(filePath, fileName),"a+") 
file.write(str(typeH))
file.write(out2.widget.result)
file.close()

#Generate model file and save

modelPath = "models/"
modelName =  user + "_svm_" + out2.widget.result + "_" + str(typeH) + ".pkl"
joblib.dump(model2, os.path.join(modelPath, modelName)) 

In [37]:
typeH

In [38]:
labelHeader

In [39]:
labelHeader + '_' + csv_file[:-4] + 'svmModel' + '.txt'